# Notebook 07: Loss Functions (losses.py)

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Derive dice_with_logits
2. Walk BCEDiceLoss and AquaBoundaryLoss
3. Understand iou_with_logits for validation
4. Explain loss weight choices

## losses.py — Every Function

### `dice_with_logits(logits, targets)`
$$\text{Dice} = \frac{2 \sum p_i t_i + \epsilon}{\sum p_i + \sum t_i + \epsilon}$$

`p_i = sigmoid(logits)` — differentiable soft Dice.

### `BCEDiceLoss`
$$L = 0.35 \cdot \text{BCEWithLogits} + 0.65 \cdot (1 - \text{Dice})$$

BCE: pixel-wise gradients. Dice: region overlap, handles imbalance.

### `AquaBoundaryLoss`
$$L_{total} = 1.0 \cdot L_{aqua} + 0.35 \cdot L_{boundary}$$

Applied independently per channel via `logits[:, 0:1]` and `logits[:, 1:2]`.

### `iou_with_logits`
Threshold at 0.5 → hard IoU — used in `validate()` for checkpoint selection.

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

In [ ]:
import torch
import torch.nn.functional as F

# Reproduce losses.py logic
def dice_with_logits(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    t = targets.float()
    inter = (probs * t).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + t.sum(dim=(2, 3))
    return ((2.0 * inter + smooth) / (union + smooth)).mean()

logits = torch.randn(2, 1, 64, 64)
targets = (torch.rand(2, 1, 64, 64) > 0.7).float()
bce = F.binary_cross_entropy_with_logits(logits, targets)
dice_loss = 1 - dice_with_logits(logits, targets)
print(f'BCE={bce:.4f}, Dice loss={dice_loss:.4f}, combined={0.35*bce + 0.65*dice_loss:.4f}')

---

## Summary

AquaBoundaryLoss = weighted BCE+Dice per head — the training objective.

**Next:** [08_Why_Dual_Heads.ipynb](08_Why_Dual_Heads.ipynb)